<a href="https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/9_aprendizaje_maquina/_assets/04_validacion_cruzada.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab" width="300"></a>

# Notebook 4 · Validación cruzada

Acompaña la página [Validación cruzada](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/separar-los-datos/validacion-cruzada/). Duración: ~35 min.

Ejecuta primero la celda de arranque. Antes de abrir cada «Juega», escribe tu
apuesta; luego mueve los controles, lee la figura y abre «Explicación».
Puedes recorrerlo por tu cuenta: la prueba permanece cerrada mientras eliges.

In [ ]:
#@title ▶ Corre esta celda primero
import os, sys, glob, base64, zlib, importlib, urllib.request
URL = os.environ.get("ML_LAB_URL",
      "https://raw.githubusercontent.com/raya-lucaria/ia_o26/main/cou"
      "rse/9_aprendizaje_maquina/_assets/ml_lab.py")
REQUIERE = "1.6"                     # la versión con que se generó este notebook
_RESPALDO = "..."

def _v(texto): return tuple(int(p) for p in texto.split("."))

def _sirve(version):                 # misma X y una Y mayor o igual
    (x, y), (x0, y0) = _v(version), _v(REQUIERE)
    return x == x0 and y >= y0

def _cargar():                       # importa ml_lab DE CERO (sin la copia en memoria)
    sys.modules.pop("ml_lab", None)
    for viejo in glob.glob("__pycache__/ml_lab.*.pyc"):
        os.remove(viejo)             # un .pyc de la copia vieja taparía la nueva
    importlib.invalidate_caches()
    import ml_lab
    return ml_lab if _sirve(ml_lab.VERSION) else None

ml = None
if os.path.exists("ml_lab.py"):      # 1) la copia local, si ya es nueva
    try: ml = _cargar()
    except Exception: ml = None
if ml is None:                       # 2) la del repositorio
    try:
        urllib.request.urlretrieve(URL, "ml_lab.py"); ml = _cargar()
    except Exception: ml = None
if ml is None:                       # 3) el respaldo: sin red o repo aún sin esta versión
    with open("ml_lab.py", "wb") as archivo:
        archivo.write(zlib.decompress(base64.b64decode(_RESPALDO)))
    ml = _cargar()
ml.requiere(REQUIERE)
ml.tema("claro")
from ml_lab import *
print(f"Listo: ml_lab {ml.VERSION}")

La validación cruzada (*cross-validation*) rota la reserva dentro del
desarrollo. Promedia las pérdidas de esas vueltas, como explica [Validación cruzada](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/separar-los-datos/validacion-cruzada/):

$$
\mathrm{CV}_k(N)=\frac{1}{k}\sum_{m=1}^{k}L_{\mathcal D_m}(\widehat\beta^{(-m)}(N)),
\qquad
\widehat N\in\operatorname*{arg\,min}_N\mathrm{CV}_k(N).
$$

Con empate, el más simple: el menor $N$. No necesitas observar un empate para aplicar la regla.

Aquí $k$ cuenta pliegues; $m$ identifica el reservado; $\widehat\beta^{(-m)}$ se ajusta sin él.
Con $k=5$, promedias cinco pérdidas. $L$ conserva su sentido de pérdida promedio.
Avisos de letras: $k$ ya no cuenta estados, simulaciones ni clases;
$m$ no indica aristas, profundidad ni tiempo medio de llegada.

**Predice.** Con $k=5$ y 48 entregas de desarrollo, ¿con cuántas entregas se ajusta cada modelo de pliegue?

Apuesta un número antes de ejecutar. Elige entre usar todas, usar solo las
reservadas o usar las restantes. Conserva «pliegues k» en 5 y recorre «vuelta m».

In [ ]:
#@title Juega: los pliegues, dibujados
import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEMILLA = 0
construir = lambda N: (lambda df, N=N: ml.ajustar(df, phi=ml.polinomio(N)))
CANDIDATOS = {N: construir(N) for N in range(13)}
DATOS = ml.PROCESOS["entregas_curvas"](n=60, semilla=SEMILLA)
DESARROLLO = pd.concat([DATOS.ent, DATOS.val])          # 48 entregas; la prueba no rota
DESARROLLO.attrs = DATOS.ent.attrs


def bloque_1(k, m):
    m = min(m, k)
    part = ml.pliegues(DESARROLLO, k=k, semilla=SEMILLA)
    ajuste, valida = part[m - 1]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    ml.dibujar_pliegues(len(DESARROLLO), k, m_activo=m, prueba=True, particion=part,
                        ax=a1)
    df_aj, df_va = DESARROLLO.iloc[ajuste], DESARROLLO.iloc[valida]
    a2.scatter(df_aj["x"], df_aj["y"], marker="o", label=f"ajusta ({len(ajuste)})")
    a2.scatter(df_va["x"], df_va["y"], marker="^", s=60,
               label=f"valida ({len(valida)})")
    modelo = construir(4)(df_aj)
    malla = pd.DataFrame({"x": np.linspace(0, 6, 300)})
    a2.plot(malla["x"], modelo.predecir(malla), label=f"grado 4 sin el pliegue {m}")
    a2.set_xlabel("x (km)"); a2.set_ylabel("y (min)"); a2.legend()
    plt.show()
    print(f"vuelta {m} de {k}: ajusta con {len(ajuste)} entregas y valida con"
          f" {len(valida)}; L del pliegue = {ml.L(modelo, df_va):.2f}")


ml.laboratorio(bloque_1, k=ml.deslizador("pliegues k", 2, 10, 1, 5),
               m=ml.deslizador("vuelta m", 1, 10, 1, 1))

**Mira.** A la izquierda, cada columna representa una entrega; la fila activa muestra qué pliegue valida. La prueba queda aparte.
A la derecha, los círculos ajustan y los triángulos validan. La curva es
$\widehat\beta^{(-m)}$: cambia al cambiar «vuelta m».

<details><summary>Explicación</summary>

Con los valores iniciales, la vuelta 1 ajusta con 38 entregas y valida con 10.
Las 48 no se reparten exactamente entre cinco pliegues: otras vueltas ajustan con 39.

Cada entrega valida una vez y participa en el ajuste de las otras vueltas.
La prueba nunca rota. La fórmula de «Validación cruzada»,
$\mathrm{CV}_k=\frac{1}{k}\sum_{m=1}^{k}L_{\mathcal D_m}(\widehat\beta^{(-m)})$,
promedia errores de modelos distintos: cada uno excluyó su pliegue de validación.
Por eso, mover «vuelta m» cambia tanto los triángulos como la curva.

</details>

**Predice.** ¿Elegirá la CV el mismo grado que el entrenamiento? ¿Y la regla de un EE, uno mayor o menor que el mínimo?

Apuesta un número para cada grado elegido. Para «un EE», admite también
«igual». Compara «mínimo» y «un EE» en «regla»; después activa «mostrar la realidad R».

In [ ]:
#@title Juega: CV contra el grado, con barras
def bloque_2(k, regla, semilla, ver_R):
    datos = ml.PROCESOS["entregas_curvas"](n=60, semilla=semilla)
    s = ml.elegir(datos, CANDIDATOS, con="cv", k=k, regla=regla,
                  orden_complejidad=list(range(13)))
    des = pd.concat([datos.ent, datos.val]); des.attrs = datos.ent.attrs
    bench = (s.cv[0].media, s.cv[0].ee)                  # N = 0: la media de ent
    R = {N: ml.riesgo(c(des), datos) for N, c in CANDIDATOS.items()} if ver_R else None
    L_ent = {N: ml.L(c(des), des) for N, c in CANDIDATOS.items()}   # como R
    ml.dibujar_cv(s, R=R, L_ent=L_ent, piso=ml.piso(datos) if ver_R else None,
                  benchmark=bench)
    plt.yscale("log")
    plt.show()
    t = s.tabla.set_index("candidato")
    N_min, N_ee = s.elegido, s.elegido_un_ee
    print(f"CV_{k}: mínimo en N = {N_min} ({t.loc[N_min, 'valor']:.2f} ±"
          f" {t.loc[N_min, 'ee']:.2f}); la regla de un EE elige N = {N_ee}")
    print(f"benchmark (N = 0, la media de entrenamiento): CV {bench[0]:.2f} ±"
          f" {bench[1]:.2f}")
    print(f"ajustes: {s.costo_ajustes} (= 13·{k} + 1)")
    if ver_R:
        mejor = min(R, key=R.get)
        print(f"con esta semilla, el mejor R es el del grado {mejor} ({R[mejor]:.2f});"
              f" el elegido tiene R = {R[s.elegido if regla == 'minimo' else N_ee]:.2f}"
              )


ml.laboratorio(bloque_2, k=ml.selector("k", [3, 5, 10], 5),
               regla=ml.selector("regla", {"mínimo": "minimo", "un EE": "un_ee"}),
               semilla=ml.deslizador("semilla", 0, 20, 1, SEMILLA),
               ver_R=ml.interruptor("mostrar la realidad R", False))

**Mira.** Los triángulos muestran medias CV; las barras, EE aproximados. Compara medias con el umbral, ambas marcas y $L_{\text{ent}}$; la línea discontinua muestra $R$.
Bajar de la raya-punto del punto de comparación (*benchmark*) es una señal.

Al mover «semilla», cambia la semilla: la población y $f$ se quedan; cambian los datos, el ajuste y, con él, su riesgo $R(\widehat\beta)$.

<details><summary>Explicación</summary>

Con los valores iniciales, ambas reglas eligen $N=6$: CV $9.69\pm1.33$; 66 ajustes. El entrenamiento prefiere 12.
Compara con el benchmark impreso: «la media de entrenamiento como punto de comparación; la clase 3 lo define».
En el despliegue, su $R=27.3$ supera 11.0 del grado 4; compara también $R$ en $N=0$.

El grado 6 se dobla en los bordes. En general, CV no favorece el grado máximo; tampoco garantiza el menor $R$.
«Validación cruzada» toma el menor $N$ cuya media cumple
$\mathrm{CV}_k(N)\le\mathrm{CV}_k(\widehat N_{\min})+\mathrm{EE}_{\mathrm{CV}}(\widehat N_{\min})$:
un grado igual o menor que el mínimo.
Las pérdidas fuera de pliegue dependen: comparar da una señal.

</details>

**Predice.** ¿Qué varía menos entre muestras: $\mathrm{CV}_5$ o $L_{\text{val}}$ con 12 entregas? ¿Siempre?

Apuesta un número para la desviación de cada estimador. Elige CV, una
validación o dispersión parecida. En «estimador», pasa de «una validación» a
«ambos» para comparar los histogramas.

In [ ]:
#@title Juega: CV contra una sola validación
def una_muestra(s):
    d = ml.PROCESOS["entregas_curvas"](n=60, semilla=s, n_realidad=2000)
    m = construir(4)(d)
    lv = np.asarray(ml.perdida(d.val["y"], m.predecir(d.val)), float)
    cv = ml.validacion_cruzada(d, construir(4), k=5)
    return {"L_val": lv.mean(), "ee_val": ml.error_estandar(lv), "cv": cv.media,
            "ee_cv": cv.ee, "R": ml.riesgo(m, d)}


T_B3 = ml.repetir(una_muestra, repeticiones=200, semilla=SEMILLA)


def bloque_3(estimador):
    series = {"una validación (12)": T_B3["L_val"], "CV_5 (48)": T_B3["cv"]}
    if estimador != "ambos":
        series = {k: v for k, v in series.items() if k.startswith(estimador)}
    ml.dibujar_histograma_estimaciones({k: v.to_numpy() for k, v in series.items()},
                                       verdadero=T_B3["R"].mean(),
                                       etiqueta="estimación")
    plt.show()
    print(f"E_D[R] ≈ {T_B3['R'].mean():.2f} (200 muestras)")
    print(f"una validación: desviación real {T_B3['L_val'].std():.2f}, EE medio"
          f" reportado {T_B3['ee_val'].mean():.2f}; 1 ajuste")
    print(f"CV_5: desviación real {T_B3['cv'].std():.2f}, EE medio reportado"
          f" {T_B3['ee_cv'].mean():.2f}; 5 ajustes")


ml.laboratorio(bloque_3, estimador=ml.selector(
    "estimador", ["una validación", "CV_5", "ambos"]))

**Mira.** Cada histograma reúne estimaciones de muestras distintas; su ancho muestra cuánto varían.
La línea vertical señala $E_{\mathcal D}[R]$ del ajuste evaluado por la validación.
Debajo, compara «desviación real» con «EE medio reportado»: una se mide entre
muestras; el otro se calcula dentro de cada muestra. Lee también el costo en ajustes.

<details><summary>Explicación</summary>

Con los valores iniciales, la desviación real es 4.85 para validación y 3.12
para CV. Sus EE medios reportados son 4.37 y 2.24, respectivamente.
Aquí CV varía menos; su EE reportado queda por debajo de su dispersión real.
Cuesta cinco ajustes, frente a uno.

La fórmula $\mathrm{CV}_k=\frac{1}{k}\sum_{m=1}^{k}L_{\mathcal D_m}(\widehat\beta^{(-m)})$
de «Validación cruzada» promedia vueltas que comparten datos de ajuste.
Sus pérdidas dependen entre sí; las barras no son intervalos de confianza exactos.
La observación sobre el EE de CV explica esa cautela.
Con otros procesos cambia la ganancia: CV no tiene siempre menor varianza.

</details>

**Predice.** En el caso 16, ¿qué grado tendrá un pliegue con error enorme: el 1 o el 8?

Apuesta un número para el mayor error por pliegue. Elige grado 1, grado 8 o
ambos. Conserva «datos» en «caso 16» y «estadístico» en «pérdida cuadrática».

In [ ]:
#@title Juega: mirar dentro de los pliegues
UMBRAL_MIN = 5.0                     # minutos: «error grande»


def var_residual(y, y_gorro):        # dentro del pliegue, divisor tamaño − 1
    return float(np.var(np.asarray(y) - np.asarray(y_gorro), ddof=1))


def prop_grandes(y, y_gorro):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(y_gorro)) > UMBRAL_MIN))


ESTADISTICOS = {"pérdida cuadrática": None, "varianza del residual": var_residual,
                f"proporción de errores > {UMBRAL_MIN:g} min": prop_grandes}
DATOS_B4 = {"entregas": DATOS, "caso 16": ml.caso(16)}


def bloque_4(estadistico, cual):
    f = ESTADISTICOS[estadistico]
    res = {N: ml.validacion_cruzada(DATOS_B4[cual], construir(N), k=5, estadistico=f)
           for N in (0, 1, 4, 8)}                    # N = 0: el benchmark
    ml.dibujar_cajas_pliegues(res, estadistico=f is not None)
    plt.yscale("log" if f is None else "linear")
    plt.show()
    for N, r in res.items():
        v = r.por_pliegue if f is None else r.por_pliegue_estadistico
        print(f"grado {N}: " + ", ".join(f"{x:.2f}" for x in v)
              + f"   (media {np.mean(v):.2f})")


ml.laboratorio(bloque_4, estadistico=ml.selector("estadístico", list(ESTADISTICOS)),
               cual=ml.selector("datos", list(DATOS_B4), "caso 16"))

**Mira.** Cada triángulo es el resultado de un pliegue; cada caja reúne los de un grado.
Busca los triángulos que se escapan y el rótulo del pliegue 3. Compara los
grados 1, 4 y 8 con el grado 0, el benchmark. Cambia «estadístico»: la altura pasa a representar otra cantidad,
no necesariamente una pérdida.

<details><summary>Explicación</summary>

Con los valores iniciales, el grado 8 se dispara en los pliegues 2 y 3.
El grado 1 tiene pérdidas por pliegue entre 11.33 y 18.61.
La caja del grado 0 muestra el benchmark en los mismos pliegues.
Ambos fallan en la zona escasamente cubierta: el pliegue 2 interpola y el 3 extrapola; la figura rotula el 3.

En «Validación cruzada», $e_i=y_i-\widehat y_i$ y $\ell_i=e_i^2$ conectan residual y pérdida cuadrática.
La varianza usa residuales con signo y divide entre el tamaño del pliegue menos uno.
La proporción cuenta errores absolutos mayores que `UMBRAL_MIN`.
Cada estadístico se calcula dentro del pliegue; después se promedian los resultados por igual.

</details>

**Predice.** Con 6 positivos y 5 pliegues al azar, ¿qué tan seguido queda algún pliegue sin positivos? ¿Y estratificando?

Apuesta un número porcentual para cada método: nunca, pocas veces o casi
siempre. Cambia «pliegues» entre «al azar» y «estratificados»; conserva
«positivos» en 6 y «k» en 5.

In [ ]:
#@title Juega: estratificar
def vacios(df, k, como, particiones=1000):
    y = df["y"].to_numpy()
    parts = (ml.pliegues(df, k=k, como=como, semilla=s) for s in range(particiones))
    return np.mean([any(y[v].sum() == 0 for _, v in p) for p in parts])


def bloque_5(positivos, k, como):
    y = np.r_[np.ones(positivos, int), np.zeros(60 - positivos, int)]
    df = pd.DataFrame({"y": y})
    part = ml.pliegues(df, k=k, como=como, semilla=SEMILLA)
    ml.dibujar_pliegues(60, k, prueba=False, y=df["y"], particion=part)
    plt.show()
    print("positivos por pliegue: " + ", ".join(str(int(y[v].sum())) for _, v in part))
    print(f"en 1000 particiones: al azar {vacios(df, k, 'azar'):.1%}, estratificada"
          f" {vacios(df, k, 'estratificado'):.1%} con algún pliegue sin positivos")


ml.laboratorio(bloque_5, positivos=ml.deslizador("positivos", 1, 12, 1, 6),
               k=ml.selector("k", [3, 5, 10], 5),
               como=ml.selector("pliegues", {"al azar": "azar",
                                             "estratificados": "estratificado"}))

**Mira.** Cada columna es una observación; el color distingue su etiqueta y la fila muestra dónde valida.
Cuenta positivos en cada pliegue de esta partición. Debajo aparecen las
fracciones de particiones con algún pliegue vacío de positivos: compara ambos
métodos, no solo el dibujo de una semilla.

<details><summary>Explicación</summary>

Con los valores iniciales, las fracciones son 88 % al azar y 0 % estratificando.
La partición al azar con semilla 0 reparte los positivos como 2, 2, 1, 0, 1.

Estratificar garantiza positivos en cada pliegue mientras haya al menos tantos
positivos como pliegues. Si $\text{positivos}<k$, necesariamente alguno queda
vacío: es el principio del palomar de «Validación cruzada».
Prueba 3 positivos con $k=5$: ningún método puede llenar todos. Baja $k$.
Si el despliegue exige separar por tiempo o grupos, esa estructura manda;
conservar proporciones no basta para decidir la separación.

</details>

**Predice.** La $\mathrm{CV}_5$ del grado ganador, ¿sobreestima, acierta o subestima el error del procedimiento entrenado con el mismo tamaño?

Apuesta un número para la diferencia promedio. Elige positiva, cero o negativa.
En «repeticiones», compara 10, 20 y 30. El precálculo se hace una vez;
el control muestra una parte de sus resultados.

In [ ]:
#@title Juega: CV anidada, en corto
CANDS_B6 = {N: construir(N) for N in range(9)}


def una_anidada(s):
    d = ml.PROCESOS["entregas_curvas"](n=60, semilla=s, n_realidad=2000)
    sel = ml.elegir(d, CANDS_B6, con="cv", k=5)
    an = ml.cv_anidada(d, CANDS_B6, k_externo=5, k_interno=5)
    des = pd.concat([d.ent, d.val]); des.attrs = d.ent.attrs
    rng = np.random.default_rng(s)
    sub = des.iloc[np.sort(rng.choice(len(des), size=len(des) * 4 // 5, replace=False))]
    sub.attrs = des.attrs
    proc = ml.elegir(sub, CANDS_B6, con="cv", k=5)
    cv_ganador = sel.tabla.set_index("candidato").loc[sel.elegido, "valor"]
    return {"CV del ganador": cv_ganador, "anidada": an.media,
            "R del procedimiento (4/5)": ml.riesgo(proc.modelo_final, d),
            "R del modelo final": ml.riesgo(sel.modelo_final, d),
            "R del benchmark": ml.riesgo(CANDS_B6[0](des), d)}


T_B6 = ml.repetir(una_anidada, repeticiones=30, semilla=SEMILLA)   # ~30 s, una vez


def bloque_6(repeticiones):
    t = T_B6.iloc[:repeticiones].drop(columns=["repeticion", "semilla"])
    medias, ee = t.mean(), t.std(ddof=1) / np.sqrt(len(t))
    plt.errorbar(range(len(medias)), medias, yerr=2 * ee, fmt="o", capsize=4)
    plt.xticks(range(len(medias)), [c.replace(" (", "\n(").replace("del ", "del\n")
                                    for c in t.columns], fontsize=8)
    plt.axvline(2.5, ls=":", color="gray")
    plt.ylabel("pérdida promedio"); plt.show()
    for c in t.columns:
        print(f"{c}: {medias[c]:.2f} ± {2 * ee[c]:.2f} (2 EE de Monte Carlo)")


ml.laboratorio(bloque_6, repeticiones=ml.selector("repeticiones", [10, 20, 30], 30))

**Mira.** Cada punto promedia repeticiones; las barras muestran dos EE de Monte Carlo.
Compara las tres primeras marcas al tamaño aproximado de una vuelta externa.
La cuarta usa todo el desarrollo; la quinta muestra «R del benchmark».
Los riesgos vienen del visor de la realidad, `datos.despliegue`: solo existe en el simulador.

<details><summary>Explicación</summary>

Con los valores iniciales, 30 repeticiones y grados de 0 a 8, la CV del ganador
promedia 10.00; la anidada, 11.79. El $R$ del procedimiento a 4/5 promedia
12.69; el del modelo final, 11.43.
La quinta columna, «R del benchmark», evalúa la media ajustada con todo el desarrollo.

En promedio, elegir el mínimo favorece estimaciones bajas por azar:
$E[L_{\text{val}}]\le E[R]$ expresa el efecto de selección en «Entrenamiento y prueba».
La anidada evalúa la elección completa sin usar el pliegue externo para elegir.
Estima el riesgo del procedimiento al tamaño de una vuelta externa.
El modelo final usa más datos y suele mejorar: selección y tamaño son efectos distintos.

</details>

**Taller.** Ejecuta la celda visible. Después copia una alternativa comentada
y deja la llamada a `ml.taller` al final.

- Con otra $f$, cambia la forma del proceso y observa si cambia el grado elegido.
- Con el caso 16, busca los pliegues que traen una entrega lejana.
- Con los mismos datos, compara `k=3` y `k=10`: anota la elección y el costo en ajustes.

Conserva `que="elegir"` y `regla="un_ee"` para comparar la misma regla.
La última asignación a `datos` determina qué ejemplo usa el taller.

In [ ]:
datos = ml.PROCESOS["entregas_curvas"](n=60, semilla=0)
# 1) otra f:   datos = ml.proceso(n=60, x={"x": ml.uniforme(0, 6)},
#                                 f=lambda d: 10 + 12 / (1 + d.x))
# 2) caso 16:  datos = ml.caso(16)
# 3) k chico contra k grande: cambia k=10 por k=3
ml.taller(datos, clase=2, que="elegir", k=10, regla="un_ee")

**Galería.** Abre los casos 13–16 como misterios. Lee la pregunta de cada caso,
anota tu apuesta y busca una pista visual antes de abrir la explicación.

En el caso 13, compara $L_{\text{prueba}}$ con $R$: el polinomio usa solo la distancia, sin tiempo, y la prueba al azar casi no engaña.
Contrasta lo que ves con la explicación: la página muestra el engaño con vecinos en distancia y tiempo.
En el caso 16, busca en qué pliegues distintos caen las dos entregas lejanas y cuánto sube el error de cada uno.
En cada misterio, identifica qué decisión cambió y qué comparación permite juzgarla.

In [ ]:
#@title Galería: casos que se rompen a la vista
ml.galeria(clase=2)

**Ejercicio.**

Cinco pliegues dan $L_{\mathcal D_m}=9.2,\ 9.0,\ 9.1,\ 8.7,\ 8.8$, en minutos cuadrados.
(a) Calcula $\mathrm{CV}_5$. (b) Calcula $\mathrm{EE}_{\mathrm{CV}}$.

**Pista:** ¿Con qué divisor va la desviación de cinco números?

<details><summary>Explicación</summary>

**(a)** $\mathrm{CV}_5=8.96$. Las desviaciones respecto de esa media son $0.24,0.04,0.14,-0.26,-0.16$.

**(b)** Sus cuadrados suman $0.172$. La desviación es $s=\sqrt{0.172/4}\approx0.207$; por tanto, $\mathrm{EE}_{\mathrm{CV}}\approx0.093$.
El divisor de la varianza muestral es cuatro; el EE divide después la desviación entre $\sqrt5$.

</details>

La CV rota la validación dentro del desarrollo y permite elegir el grado sin
abrir la prueba. Con los valores iniciales, ambas reglas eligieron grado 6;
eso no garantizó el menor riesgo. Mira también los pliegues y distingue la
evaluación del procedimiento de la del modelo final.

La pregunta que sigue es: ¿de dónde sale la U?